# Knowledge Graph Index — SPO triple extraction + graph traversal

Vector search answers *what* documents are relevant. A **knowledge graph index** answers *how* entities relate. Key differences:

| Concern | Vector index | KG index |
|---|---|---|
| Unit | documents | entities + relations |
| Strength | semantic similarity | multi-hop reasoning |
| Query | "papers about speculative decoding" | "how does AdaSpec relate to Llama-3 70B?" |

This notebook: LLM extracts (subject, predicate, object) triples from each abstract → stores in a `networkx.DiGraph` → answers multi-hop questions via shortest-path traversal + LLM.

In [ ]:
import os, sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / 'shared').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if not (os.getenv('OPENAI_API_KEY') or os.getenv('ANTHROPIC_API_KEY')):
    os.environ.setdefault('LLM_CACHE_ONLY', '1')
print('LLM_CACHE_ONLY =', os.environ.get('LLM_CACHE_ONLY', '0'))


In [ ]:
import json
import networkx as nx
from shared.llm import Message, complete
from shared.loaders import load_corpus, load_golden_qa

MODEL = 'openai/gpt-4o-mini'
NS = '02-indexing/04-knowledge-graph-index'
DOCS = load_corpus()
QA = {q.id: q for q in load_golden_qa()}

## Step 1 — extract SPO triples (one LLM call per abstract)

In [ ]:
EXTRACT_SYS = (
    'Extract (subject, predicate, object) triples from the given abstract. '
    'Return ONLY a JSON array of arrays: [[subject, predicate, object], ...]. '
    'Use short noun phrases. Extract 3-5 triples. No prose.'
)

def extract_triples(abstract):
    raw = complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=EXTRACT_SYS),
        Message(role='user', content=abstract),
    ]).content
    return json.loads(raw)

per_doc = {d.arxiv_id: extract_triples(d.abstract) for d in DOCS}
for did, triples in list(per_doc.items())[:2]:
    print(f'{did}:')
    for t in triples:
        print(f'  {t[0]} --[{t[1]}]--> {t[2]}')

## Step 2 — build the directed knowledge graph

In [ ]:
G = nx.DiGraph()
for did, triples in per_doc.items():
    for s, p, o in triples:
        G.add_edge(s, o, predicate=p, doc=did)

print('nodes:', G.number_of_nodes(), ' edges:', G.number_of_edges())
print('Example edges:')
for s, o, data in list(G.edges(data=True))[:5]:
    print(f'  {s} --[{data["predicate"]}]--> {o}')

## Step 3 — multi-hop query via shortest path + LLM

In [ ]:
PATH_SYS = (
    'Given a list of knowledge-graph triples describing a path from source to target, '
    'produce a concise one-sentence answer to the question. Use ONLY the triples provided.'
)

def kg_answer(q, source, target):
    try:
        path = nx.shortest_path(G, source=source, target=target)
    except (nx.NetworkXNoPath, nx.NodeNotFound):
        return 'No path found in the knowledge graph.'
    triples = []
    for a, b in zip(path, path[1:]):
        data = G.get_edge_data(a, b, default={})
        triples.append(f'[{repr(a)}, {repr(data.get("predicate","related to"))}, {repr(b)}]')
    path_str = ' -> '.join(triples)
    return complete(model=MODEL, namespace=NS, messages=[
        Message(role='system', content=PATH_SYS),
        Message(role='user', content=f'Question: {q}\n\nTriple path:\n{path_str}'),
    ]).content

# Multi-hop: q23 = RA-MoE parameters + routing
q23 = QA['q23'].question
print('Q:', q23)
print(kg_answer(q23, 'RA-MoE', 'p99 decode latency by 38%'))

# Multi-hop: q24 = HA-Retrieve chunking + attention mechanism
q24 = QA['q24'].question
print('\nQ:', q24)
print(kg_answer(q24, 'HA-Retrieve', 'recall@10 by 17 points over BGE-large'))

## What you can extend

* Add entity resolution: merge `RA-MoE`, `RA MoE`, `Routing-Aware Sparse MoE` into one node.
* Use a vector index over entity names as an entry point: fuzzy search for the right `source` node.
* Combine with GraphRAG (`01-rag/09-graph-rag`): that leaf builds a community-level summary graph; this one builds a fact-level triple graph — they're complementary.
* For production: Neo4j (`py2neo` or `neo4j` driver) with Cypher MATCH paths.